# Train on a free Colab GPU

Works with any dataset exported by `backend/scripts/export_for_colab.py` (run
`python scripts/export_for_colab.py <dataset_name>` from `backend/`, or use
the "Suggest cylinders"/labeling tools in the app first, then export).

**Before running:** `Runtime` menu → `Change runtime type` → select **T4 GPU** → Save.

Then: `Runtime` → `Run all`. When it gets to the upload cell, upload your `<dataset_name>-dataset.zip`.

At the end, `trained-model.zip` (containing `best.pt`) downloads automatically —
that's what you bring back to **Detect → Import a model** (desktop app or web app).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
!pip install -q ultralytics

In [ ]:
from google.colab import files

print("Upload your <dataset_name>-dataset.zip:")
uploaded = files.upload()
zip_names = [n for n in uploaded if n.endswith(".zip")]
assert zip_names, "Expected a .zip file (from export_for_colab.py)"
dataset_zip = zip_names[0]
print("Using:", dataset_zip)

In [ ]:
import shutil

shutil.unpack_archive(dataset_zip, "dataset")
!echo '--- dataset/data.yaml ---'; cat dataset/data.yaml
!echo '--- image counts ---'; find dataset/images -type f | wc -l

## Train

Fine-tunes pretrained `yolov8n-seg` (matches what the desktop/web app does
locally, just on a GPU instead of CPU — labels are polygon outlines, so the
app always trains the segmentation variant, even for plain box labels).
Adjust `epochs`/`imgsz`/`batch` as needed; watch the `val` mask mAP in the
output and stop early (interrupt the cell) if it plateaus.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")
results = model.train(
    data="dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    project="runs",
    name="train",
    exist_ok=True,
)

In [ ]:
# quick sanity check on the validation set
metrics = model.val()
print(metrics.seg.map, "(mask mAP50-95)")

In [ ]:
import shutil
from google.colab import files

shutil.copy("runs/train/weights/best.pt", "best.pt")
shutil.make_archive("trained-model", "zip", ".", "best.pt")
files.download("trained-model.zip")